In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import socket
import numpy as np

c:\Users\roger\AppData\Local\Programs\Python\Python314\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
# 1. Define a Lightweight CNN Model
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input: 1 x 28 x 28
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10) # 10 output classes (digits 0-9)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)

        x = self.conv2(x)
        x = self.relu(x)
        x = self.pool(x)

        x = x.view(x.size(0), -1)  # Flatten

        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)
        return x

In [20]:
def train_and_save_model():
    print("Downloading MNIST and training CNN model...")
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
    
    train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    model = SimpleCNN()
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.01)

    # Quick 5-epoch training loop for demonstration purposes
    model.train()
    for _ in range(10):
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            if batch_idx % 200 == 0:
                print(f"Training Batch {batch_idx}/{len(train_loader)} - Loss: {loss.item():.4f}")
            if batch_idx >= 300:  # Cap at 300 batches for a fast setup
                break

    print("Training complete! Saving weights...")
    #torch.save(model.state_dict(), "mnist_cnn_weights.pth")
    return model

In [18]:
# Train or load model
model = train_and_save_model()

Training Batch 0/938 - Loss: 2.3017
Training Batch 200/938 - Loss: 0.1869
Training Batch 0/938 - Loss: 0.1129
Training Batch 200/938 - Loss: 0.1068
Training Batch 0/938 - Loss: 0.1413
Training Batch 200/938 - Loss: 0.1264
Training Batch 0/938 - Loss: 0.1166
Training Batch 200/938 - Loss: 0.0569
Training Batch 0/938 - Loss: 0.0349
Training Batch 200/938 - Loss: 0.0736
Training Batch 0/938 - Loss: 0.0612
Training Batch 200/938 - Loss: 0.1087
Training Batch 0/938 - Loss: 0.0377
Training Batch 200/938 - Loss: 0.2048
Training Batch 0/938 - Loss: 0.0200
Training Batch 200/938 - Loss: 0.1572
Training Batch 0/938 - Loss: 0.0976
Training Batch 200/938 - Loss: 0.0675
Training Batch 0/938 - Loss: 0.1547
Training Batch 200/938 - Loss: 0.0605
Training complete! Saving weights...


In [19]:
# Save Weights
total_bytes = 0

with open("weights.bin", "wb") as f:
    # Iterate through all named parameters in the network
    for name, param in model.named_parameters():
        if "weight" in name or "bias" in name:
            # 1. Detach from graph and move to CPU numpy
            w_np = param.detach().cpu().numpy()
            
            # 2. Quantize float32 weights down to int8 for FPGA/C compatibility
            # (Scaling to fit the -128 to 127 range)
            w_min, w_max = w_np.min(), w_np.max()
            scale = max(abs(w_min), abs(w_max))
            if scale == 0: 
                scale = 1.0
            
            w_int8 = np.clip(np.round(w_np / scale * 127), -128, 127).astype(np.int8)
            
            # 3. Write raw bytes sequentially into the single file
            f.write(w_int8.tobytes())
            
            layer_size = w_int8.size
            total_bytes += layer_size
            print(f"Packed {name} | Shape: {w_np.shape} | Bytes: {layer_size}")

Packed conv1.weight | Shape: (32, 1, 3, 3) | Bytes: 288
Packed conv1.bias | Shape: (32,) | Bytes: 32
Packed conv2.weight | Shape: (64, 32, 3, 3) | Bytes: 18432
Packed conv2.bias | Shape: (64,) | Bytes: 64
Packed fc1.weight | Shape: (128, 3136) | Bytes: 401408
Packed fc1.bias | Shape: (128,) | Bytes: 128
Packed fc2.weight | Shape: (10, 128) | Bytes: 1280
Packed fc2.bias | Shape: (10,) | Bytes: 10


In [12]:
def main():
    UDP_IP = "0.0.0.0"  # Listen on all local interfaces
    UDP_PORT = 8080

    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    
    # CRITICAL: Allow immediate reuse of the port to prevent WinError 10048
    sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    loopNum = 0

    model.eval()
    try:
        sock.bind((UDP_IP, UDP_PORT))
        print(f"\nLaptop inference server active. Listening for Raspberry Pi stream on port {UDP_PORT}...")

        with torch.no_grad():
            while True:
                data, addr = sock.recvfrom(1024)
                loopNum += 1

                if len(data) == 784:
                    img_array = np.frombuffer(data, dtype=np.uint8).astype(np.float32)
                    tensor_img = torch.tensor(img_array).view(1, 1, 28, 28) / 255.0
                    tensor_img = (tensor_img - 0.1307) / 0.3081

                    outputs = model(tensor_img)
                    probabilities = torch.softmax(outputs, dim=1)
                    predicted_digit = torch.argmax(outputs, dim=1).item()
                    confidence = probabilities[0][predicted_digit].item() * 100

                    if (confidence > 80):
                        print(f"Received frame from {addr[0]} -> Predicted Digit: {predicted_digit} (Confidence: {confidence:.2f}%)")
                        loopNum = 0
                else:
                    print(f"Received malformed packet size: {len(data)} bytes (Expected 784)")

    except KeyboardInterrupt:
        print("\nShutting down server gracefully...")
    finally:
        sock.close()
        print("Socket closed and port successfully released.")

In [13]:
if __name__ == "__main__":
    main()


Laptop inference server active. Listening for Raspberry Pi stream on port 8080...
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 86.45%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 90.35%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 82.03%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 86.52%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 93.41%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 83.28%)
Received frame from 192.168.10.1 -> Predicted Digit: 8 (Confidence: 81.64%)
Received frame from 192.168.10.1 -> Predicted Digit: 0 (Confidence: 81.40%)
Received frame from 192.168.10.1 -> Predicted Digit: 0 (Confidence: 80.11%)
Received frame from 192.168.10.1 -> Predicted Digit: 6 (Confidence: 81.73%)
Received frame from 192.168.10.1 -> Predicted Digit: 6 (Confidence: 93.17%)
Received frame from 192.168.10.1 -> Predicted Digit: 6 (Confidence: 82.70%)
Recei